# Task 3: A/B Hypothesis Testing

## Objective

The objective of this analysis is to statistically validate whether significant differences exist across demographic and geographic insurance segments.

This analysis evaluates:

- Claim Frequency
- Claim Severity
- Margin (Profitability)

The findings will support data-driven pricing, segmentation, and underwriting strategies.

In [2]:
!pip install scipy

  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl (36.5 MB)



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind, chi2_contingency

pd.set_option("display.max_columns", None)

# 1. Load and Inspect Dataset

This section loads the cleaned insurance dataset and performs initial inspection to verify structure, column names, and data types before statistical testing.

In [8]:
df = pd.read_csv(
    "../data/insurance_data_clean.csv",
    low_memory=False, sep="|"
)

df.head()
df.info()
df.columns.tolist()

<class 'pandas.DataFrame'>
RangeIndex: 973382 entries, 0 to 973381
Data columns (total 52 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   UnderwrittenCoverID       973382 non-null  int64  
 1   PolicyID                  973382 non-null  int64  
 2   TransactionMonth          973382 non-null  str    
 3   IsVATRegistered           973382 non-null  bool   
 4   Citizenship               973382 non-null  str    
 5   LegalType                 973382 non-null  str    
 6   Title                     973382 non-null  str    
 7   Language                  973382 non-null  str    
 8   Bank                      831862 non-null  str    
 9   AccountType               934647 non-null  str    
 10  MaritalStatus             965123 non-null  str    
 11  Gender                    963846 non-null  str    
 12  Country                   973382 non-null  str    
 13  Province                  973382 non-null  str    
 14 

['UnderwrittenCoverID',
 'PolicyID',
 'TransactionMonth',
 'IsVATRegistered',
 'Citizenship',
 'LegalType',
 'Title',
 'Language',
 'Bank',
 'AccountType',
 'MaritalStatus',
 'Gender',
 'Country',
 'Province',
 'PostalCode',
 'MainCrestaZone',
 'SubCrestaZone',
 'ItemType',
 'mmcode',
 'VehicleType',
 'RegistrationYear',
 'make',
 'Model',
 'Cylinders',
 'cubiccapacity',
 'kilowatts',
 'bodytype',
 'NumberOfDoors',
 'VehicleIntroDate',
 'CustomValueEstimate',
 'AlarmImmobiliser',
 'TrackingDevice',
 'CapitalOutstanding',
 'NewVehicle',
 'WrittenOff',
 'Rebuilt',
 'Converted',
 'CrossBorder',
 'NumberOfVehiclesInFleet',
 'SumInsured',
 'TermFrequency',
 'CalculatedPremiumPerTerm',
 'ExcessSelected',
 'CoverCategory',
 'CoverType',
 'CoverGroup',
 'Section',
 'Product',
 'StatutoryClass',
 'StatutoryRiskType',
 'TotalPremium',
 'TotalClaims']

# 2. Define Key Performance Indicators (KPIs)

The following KPIs are used to measure insurance risk and profitability:

- Claim Frequency → whether a policy had at least one claim
- Claim Severity → average claim amount
- Margin → TotalPremium − TotalClaims

In [9]:
df["HasClaim"] = df["TotalClaims"] > 0

df["Margin"] = (
    df["TotalPremium"] - df["TotalClaims"]
)

df[["HasClaim", "Margin"]].head()

,HasClaim,Margin
0,False,21.929825
1,False,21.929825
2,False,0.000000
3,False,512.848070
4,False,0.000000


# 3. Statistical Test Functions

Reusable functions are implemented for:

- Chi-Square Test (categorical outcomes)
- Independent T-Test (numerical comparisons)

These functions support consistent and reusable hypothesis testing across multiple business questions.

In [10]:
def claim_frequency_test(df, group_col, group_a, group_b):

    subset = df[df[group_col].isin([group_a, group_b])]

    contingency = pd.crosstab(
        subset[group_col],
        subset["HasClaim"]
    )

    chi2, p, dof, expected = chi2_contingency(contingency)

    return p


def numerical_test(df, group_col, group_a, group_b, metric):

    subset = df[df[group_col].isin([group_a, group_b])]

    group1 = subset[
        subset[group_col] == group_a
    ][metric]

    group2 = subset[
        subset[group_col] == group_b
    ][metric]

    stat, p = ttest_ind(
        group1,
        group2,
        equal_var=False
    )

    return p

# 4. Hypothesis Test: Risk Differences Across Provinces

## Null Hypothesis (H₀)

There are no significant risk differences across provinces.

## KPI

Claim Severity

## Statistical Test

Independent Two-Sample T-Test

In [12]:
province_p = numerical_test(
    df,
    group_col="Province",
    group_a="Gauteng",
    group_b="Western Cape",
    metric="TotalClaims"
)

province_p

if province_p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


# 5. Hypothesis Test: Risk Differences Between Zip Codes

## Null Hypothesis (H₀)

There are no significant risk differences between zip codes.

## KPI

Claim Frequency

## Statistical Test

Chi-Square Test

In [13]:
zip_risk_p = claim_frequency_test(
    df,
    group_col="PostalCode",
    group_a=2000,
    group_b=8000
)

zip_risk_p

if zip_risk_p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


# 6. Hypothesis Test: Margin Differences Between Zip Codes

## Null Hypothesis (H₀)

There is no significant profitability difference between zip codes.

## KPI

Margin = TotalPremium − TotalClaims

## Statistical Test

Independent Two-Sample T-Test

In [14]:
margin_p = numerical_test(
    df,
    group_col="PostalCode",
    group_a=2000,
    group_b=8000,
    metric="Margin"
)

margin_p
if margin_p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


# 7. Hypothesis Test: Risk Difference Between Women and Men

## Null Hypothesis (H₀)

There is no significant risk difference between Women and Men.

## KPI

Claim Severity

## Statistical Test

Independent Two-Sample T-Test

In [15]:
gender_p = numerical_test(
    df,
    group_col="Gender",
    group_a="Male",
    group_b="Female",
    metric="TotalClaims"
)

gender_p

if gender_p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


# 8. Hypothesis Testing Summary

The following table summarizes:

- The hypothesis tested
- The statistical test used
- The p-value
- Final decision

In [16]:
results = pd.DataFrame({

    "Hypothesis": [
        "Province Risk Difference",
        "Zip Code Risk Difference",
        "Zip Code Margin Difference",
        "Gender Risk Difference"
    ],

    "Test": [
        "T-Test",
        "Chi-Square",
        "T-Test",
        "T-Test"
    ],

    "P-Value": [
        province_p,
        zip_risk_p,
        margin_p,
        gender_p
    ]
})

results["Decision"] = np.where(
    results["P-Value"] < 0.05,
    "Reject H0",
    "Fail to Reject H0"
)

results

,Hypothesis,Test,P-Value,Decision
0,Province Risk Difference,T-Test,0.085224,Fail to Reject H0
1,Zip Code Risk Difference,Chi-Square,0.279775,Fail to Reject H0
2,Zip Code Margin Difference,T-Test,0.836238,Fail to Reject H0
3,Gender Risk Difference,T-Test,0.766966,Fail to Reject H0


# 9. Business Recommendations

## Province-Level Risk

If the null hypothesis is rejected, regional risk varies significantly across provinces. This suggests geographic premium adjustments may improve underwriting accuracy.

## Zip Code Risk

Significant zip code differences indicate localized risk behavior and justify location-based segmentation strategies.

## Margin Differences

Profitability differences across zip codes may reveal underpriced or overpriced customer segments.

## Gender-Based Risk

If significant differences exist between genders, demographic variables may influence claim behavior and portfolio risk.

In [17]:
results

,Hypothesis,Test,P-Value,Decision
0,Province Risk Difference,T-Test,0.085224,Fail to Reject H0
1,Zip Code Risk Difference,Chi-Square,0.279775,Fail to Reject H0
2,Zip Code Margin Difference,T-Test,0.836238,Fail to Reject H0
3,Gender Risk Difference,T-Test,0.766966,Fail to Reject H0


# 10. Export Results

The final hypothesis testing summary is exported for reporting and documentation purposes.

In [18]:
results.to_csv(
    "../reports/hypothesis_test_results.csv",
    index=False
)

print("Results exported successfully.")

Results exported successfully.
